In [ ]:
library(patchwork)
library(tidyverse)
library(here)
library(survival)


In [ ]:
# source oof calibration helper
source(here("helpers/oof_calibration.r"))

## Calibration

Fully out-of-fold calibration at 1, 3, and 5 years. Edit the specs, then run the helper, load, compute, and figure cells. COPD is included in the calibration summary and saved as its own figure.


In [ ]:
# Calibration at 1, 3, and 5 years from fold-specific OOF absolute risks.
risk_dir <- here("outputs/main/absolute_risks")

horizons <- c(1, 3, 5)
n_groups <- 10
verbose_calibration <- FALSE

calib_palette <- c(
  "SOC"     = "#E64B35",
  "SOC+NMR" = "#4DBBD5"
)

# Primary 3x3 grid. COPD is calibrated with the same helper and saved separately.
disease_specs <- list(
  t2d = list(
    title     = "T2D Calibration",
    soc_file  = "t2d_a1c_marker_fold_baseline_risk.rds",
    nmr_file  = "t2d_a1c_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.15),
      `3` = c(0, 0.30),
      `5` = c(0, 0.40)
    )
  ),
  ckd = list(
    title     = "CKD Calibration",
    soc_file  = "ckd_egfr_marker_fold_baseline_risk.rds",
    nmr_file  = "ckd_egfr_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.15),
      `3` = c(0, 0.35),
      `5` = c(0, 0.55)
    )
  ),
  nafld = list(
    title     = "MASLD Calibration",
    soc_file  = "nafld_fib4_marker_fold_baseline_risk.rds",
    nmr_file  = "nafld_fib4_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.08),
      `3` = c(0, 0.18),
      `5` = c(0, 0.25)
    )
  ),
  chd_revasc = list(
    title     = "CHD Calibration",
    soc_file  = "chd_revasc_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file  = "chd_revasc_prevent_ingredients_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.04),
      `3` = c(0, 0.10),
      `5` = c(0, 0.15)
    )
  ),
  hf_cm = list(
    title     = "Heart Failure Calibration",
    soc_file  = "hf_cm_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file  = "hf_cm_prevent_ingredients_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.05),
      `3` = c(0, 0.14),
      `5` = c(0, 0.22)
    )
  ),
  tia_stroke = list(
    title     = "TIA/Stroke Calibration",
    soc_file  = "tia_stroke_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file  = "tia_stroke_prevent_ingredients_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.03),
      `3` = c(0, 0.07),
      `5` = c(0, 0.12)
    )
  ),
  acute_mi = list(
    title     = "Acute MI Calibration",
    soc_file  = "acute_mi_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file  = "acute_mi_prevent_ingredients_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.02),
      `3` = c(0, 0.05),
      `5` = c(0, 0.10)
    )
  ),
  total_cvd = list(
    title     = "Total CVD Calibration",
    soc_file  = "total_cvd_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file  = "total_cvd_prevent_ingredients_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.07),
      `3` = c(0, 0.20),
      `5` = c(0, 0.30)
    )
  ),
  afib = list(
    title     = "Afib Calibration",
    soc_file  = "afib_chargeaf_ingredients_marker_fold_baseline_risk.rds",
    nmr_file  = "afib_chargeaf_ingredients_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.04),
      `3` = c(0, 0.12),
      `5` = c(0, 0.20)
    )
  ),
  copd = list(
    title     = "COPD Calibration",
    soc_file  = "copd_smoking_ldl_marker_fold_baseline_risk.rds",
    nmr_file  = "copd_smoking_ldl_marker_nmr_fold_baseline_risk.rds",
    axis_limits = list(
      `1` = c(0, 0.03),
      `3` = c(0, 0.07),
      `5` = c(0, 0.105)
    )
  )
)

grid_layout <- list(
  c("t2d", "ckd", "nafld"),
  c("chd_revasc", "hf_cm", "tia_stroke"),
  c("acute_mi", "total_cvd", "afib")
)

legend_id <- "total_cvd"


In [ ]:
oof_risk_column <- function(t_star) {
  paste0("pred_risk_", as.integer(t_star), "yr")
}

load_disease_oof_risks <- function(spec, risk_dir) {
  soc_path <- file.path(risk_dir, spec$soc_file)
  nmr_path <- file.path(risk_dir, spec$nmr_file)
  missing <- c(soc_path, nmr_path)[!file.exists(c(soc_path, nmr_path))]
  if (length(missing) > 0) {
    stop("Missing absolute-risk RDS file(s):\n", paste(missing, collapse = "\n"))
  }
  list(
    soc = readRDS(soc_path),
    nmr = readRDS(nmr_path)
  )
}

plot_disease_oof_calibration <- function(
  spec,
  oof_objects,
  t_star,
  n_groups = 10,
  palette,
  verbose = FALSE
) {
  risk_col <- oof_risk_column(t_star)
  horizon_key <- as.character(t_star)
  axis_limits <- spec$axis_limits[[horizon_key]]

  if (is.null(axis_limits)) {
    stop(
      "No axis_limits for '", spec$title, "' at t_star = ", t_star,
      ". Add disease_specs[[id]]$axis_limits$`", t_star, "`."
    )
  }

  soc_oof <- oof_objects$soc$oof_predictions
  nmr_oof <- oof_objects$nmr$oof_predictions
  if (!risk_col %in% names(soc_oof) || !risk_col %in% names(nmr_oof)) {
    stop(
      "Column `", risk_col, "` is missing for ", spec$title,
      ". Re-generate absolute risks with prediction_times including ", t_star, "."
    )
  }

  cal_soc <- compute_calibration_df_from_oof_risk(
    oof_objects$soc,
    t_star   = t_star,
    risk_col = risk_col,
    n_groups = n_groups,
    verbose  = verbose
  )
  cal_nmr <- compute_calibration_df_from_oof_risk(
    oof_objects$nmr,
    t_star   = t_star,
    risk_col = risk_col,
    n_groups = n_groups,
    verbose  = verbose
  )

  p <- plot_calibration_compare(
    list("SOC" = cal_soc, "SOC+NMR" = cal_nmr),
    title           = spec$title,
    t_star          = t_star,
    axis_limits     = axis_limits,
    show_error_bars = TRUE,
    show_lines      = TRUE,
    palette         = palette
  )

  list(
    plot = p,
    cal_soc = cal_soc,
    cal_nmr = cal_nmr,
    risk_col = risk_col,
    axis_limits = axis_limits
  )
}

assemble_primary_grid <- function(plot_list, grid_layout, legend_id) {
  hide_legend <- function(p) p + theme(legend.position = "none")
  rows <- lapply(grid_layout, function(ids) {
    Reduce(`|`, lapply(ids, function(id) {
      if (identical(id, legend_id)) {
        plot_list[[id]] + theme(legend.position = "bottom")
      } else {
        hide_legend(plot_list[[id]])
      }
    }))
  })
  Reduce(`/`, rows)
}

In [ ]:
# Load each disease's SoC and SoC+NMR absolute-risk objects once.
missing_files <- unlist(lapply(disease_specs, function(spec) {
  paths <- file.path(risk_dir, c(spec$soc_file, spec$nmr_file))
  paths[!file.exists(paths)]
}), use.names = FALSE)

if (length(missing_files) > 0) {
  stop(
    "Missing absolute-risk RDS file(s):\n",
    paste(missing_files, collapse = "\n")
  )
}

oof_store <- lapply(disease_specs, load_disease_oof_risks, risk_dir = risk_dir)
names(oof_store) <- names(disease_specs)
cat("Loaded", length(oof_store), "diseases from\n", risk_dir, "\n")

In [ ]:
# Compute calibration for every disease x horizon.
# t_star is always paired with pred_risk_{t}yr.
calib_by_horizon <- lapply(setNames(horizons, paste0(horizons, "yr")), function(t_star) {
  disease_results <- lapply(names(disease_specs), function(id) {
    plot_disease_oof_calibration(
      spec        = disease_specs[[id]],
      oof_objects = oof_store[[id]],
      t_star      = t_star,
      n_groups    = n_groups,
      palette     = calib_palette,
      verbose     = verbose_calibration
    )
  })
  names(disease_results) <- names(disease_specs)
  list(
    t_star = t_star,
    risk_col = oof_risk_column(t_star),
    plots = lapply(disease_results, `[[`, "plot"),
    results = disease_results
  )
})

calib_ici_summary <- do.call(rbind, lapply(calib_by_horizon, function(horizon_res) {
  do.call(rbind, lapply(names(horizon_res$results), function(id) {
    res <- horizon_res$results[[id]]
    axis_vals <- c(
      res$cal_soc$pred_mean, res$cal_soc$obs_risk, res$cal_soc$obs_upper,
      res$cal_nmr$pred_mean, res$cal_nmr$obs_risk, res$cal_nmr$obs_upper
    )
    data.frame(
      disease   = id,
      title     = disease_specs[[id]]$title,
      t_star    = horizon_res$t_star,
      risk_col  = horizon_res$risk_col,
      soc_ici   = attr(res$cal_soc, "ici"),
      nmr_ici   = attr(res$cal_nmr, "ici"),
      axis_max  = res$axis_limits[2],
      suggested_axis_max = ceiling((max(axis_vals, na.rm = TRUE) + 0.01) * 20) / 20,
      stringsAsFactors = FALSE
    )
  }))
}))
rownames(calib_ici_summary) <- NULL

In [ ]:
options(repr.plot.height = 10, repr.plot.width = 10)

p_calib_primary_1yr <- assemble_primary_grid(
  calib_by_horizon[["1yr"]]$plots, grid_layout, legend_id
)
p_calib_primary_3yr <- assemble_primary_grid(
  calib_by_horizon[["3yr"]]$plots, grid_layout, legend_id
)
p_calib_primary_5yr <- assemble_primary_grid(
  calib_by_horizon[["5yr"]]$plots, grid_layout, legend_id
)

p_calib_primary_1yr
p_calib_primary_3yr
p_calib_primary_5yr

fig_dir <- here("outputs/figures")
ggsave(file.path(fig_dir, "[CALIB]_1yr.png"), p_calib_primary_1yr, width = 10, height = 10, dpi = 300)
ggsave(file.path(fig_dir, "[CALIB]_3yr.png"), p_calib_primary_3yr, width = 10, height = 10, dpi = 300)
ggsave(file.path(fig_dir, "[CALIB]_5yr.png"), p_calib_primary_5yr, width = 10, height = 10, dpi = 300)

## Supplemental Outcome: COPD

In [ ]:
# Supplemental 1/3/5-year row. COPD is in disease_specs, not grid_layout.
copd_titles <- c(
  "1yr" = "COPD Calibration, 1 yr",
  "3yr" = "COPD Calibration, 3 yr",
  "5yr" = "COPD Calibration, 5 yr"
)

options(repr.plot.height = 3.75, repr.plot.width = 10)

p_copd_all <- (
  calib_by_horizon[["1yr"]]$plots$copd + ggtitle(copd_titles[["1yr"]])
) + (
  calib_by_horizon[["3yr"]]$plots$copd + ggtitle(copd_titles[["3yr"]])
) + (
  calib_by_horizon[["5yr"]]$plots$copd + ggtitle(copd_titles[["5yr"]])
) + plot_layout(nrow = 1)

p_copd_all

ggsave(
  here("outputs/figures/COPD_CALIB.png"),
  plot = p_copd_all,
  width = 10,
  height = 3.75,
  dpi = 300
)
